# Predicting CONMEBOL World Cup qualifier scores

**DA380A Machine Learning · HT2026 · Group 28** — Lawrence Arciniegas, Gunnar Arias

**Goal:** predict the **home score and the away score** of international football matches from information known *before* kick-off, and evaluate the predictions on the **CONMEBOL World Cup qualifiers** of the 2026 cycle (the 10 South American national teams, 90 matches).

**Dataset:** Mart Jürisoo, *International football results from 1872* (Kaggle `martj42/international-football-results-from-1872-to-2017`; downloaded 13 Oct 2026 from the author's GitHub mirror — see the README).

**Algorithms:** (1) Linear Regression **implemented from scratch** with NumPy · (2) Random Forest Regressor (scikit-learn).

The notebook follows the 11-step structure of the project guidelines. The full plan is in [`docs/PLAN.md`](../docs/PLAN.md).

| Section | Step | Phase |
|---|---|---|
| 1 | Import libraries | 1 |
| 2 | Load the dataset | 1 |
| 3 | Explore the data | 1 |
| 4 | Preprocess the data (feature engineering) | 2 |
| 5 | Define features and target | 2 |
| 6 | Split into training and test sets | 4 |
| 7 | **Algorithm 1 — Linear Regression from scratch** | 3–4 |
| 8 | Algorithm 2 — Random Forest Regressor | 5 |
| 9 | Evaluate both models | 6 |
| 10 | Compare the results | 6 |
| 11 | Visualisations | 6 |

## 1. Import libraries

In [ ]:
import inspect
import sys
import time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Code
from sklearn.model_selection import TimeSeriesSplit

from src.data import (CONMEBOL, RANDOM_STATE, TARGETS, TEST_END, TEST_START, TRAIN_START,
                      clean_results, load_former_names, load_results)

from src.evaluation import (MeanBaseline, VenueBaseline, choose_draw_band, outcome_scores,
                            regression_metrics, time_series_cv)
from src.features import (ELO_K, FEATURES, FORM_WINDOW, H2H_WINDOW, VENUE_ALTITUDE_M,
                          build_features)
from src.linear_regression import LinearRegressionScratch

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Load the dataset

`results.csv` has one row per match: date, home and away team, the final score (penalty shoot-outs not included), tournament, city, country, and whether the venue was neutral.

In [ ]:
raw = load_results()
former = load_former_names()
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 3. Explore the data

### 3.1 Data quality

In [ ]:
teams = pd.unique(raw[["home_team", "away_team"]].values.ravel())
print(f"Date range: {raw['date'].min().date()} -> {raw['date'].max().date()}")
print(f"Teams: {len(teams)} | tournaments: {raw['tournament'].nunique()}")
print(f"Missing values: {raw.isna().sum().sum()}")
print(f"Fully duplicated rows: {raw.duplicated().sum()}")
print(f"Same date + same teams: {raw.duplicated(subset=['date', 'home_team', 'away_team']).sum()}")
print(f"Negative scores: {((raw['home_score'] < 0) | (raw['away_score'] < 0)).sum()}")
raw[raw.duplicated(subset=["date", "home_team", "away_team"], keep=False)]

In [ ]:
ax = raw.groupby(raw["date"].dt.year).size().plot(figsize=(12, 3.5))
ax.axvspan(TRAIN_START.year, TEST_START.year, color="tab:green", alpha=0.08, label="training window")
ax.axvspan(TEST_START.year, TEST_END.year, color="tab:red", alpha=0.12, label="test window")
ax.set(title="International matches per year", xlabel="", ylabel="Matches")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

The raw data is clean in the technical sense: no missing values, no duplicated rows, no impossible scores. The one pair with the same date and teams (Tahiti vs New Caledonia, 1974) has two *different* scores, so it is most likely two real matches on the same day; it is from before our training window and is kept.

### 3.2 Cleaning: which teams and matches we keep

1. **Former team names.** `former_names.csv` lists 36 old names (e.g. Dahomey → Benin). None of them occurs in `results.csv` — the dataset already uses one name per team — so nothing has to be renamed. We keep the check so it stays visible.
2. **Only FIFA national teams.** The data also contains regional and unrecognised teams (Catalonia, Corsica, Tibet, …) from tournaments such as the Island Games or the CONIFA World Football Cup. They produce extreme results (e.g. 22–0) that have nothing to do with our problem. We keep only teams that have played at least one **World Cup or World Cup qualifier** — a simple, data-driven definition of a FIFA member.
3. **Periods.** The full history is kept, because Phase 2 needs old matches to build team ratings, but every match gets a label: *warm-up* (before 1994, features only), *train* (1994 → Aug 2023), *test* (Sep 2023 → Sep 2025, the 2026 qualifying cycle) and *after*.

In [ ]:
df, notes = clean_results(raw, former)
print("\n".join(notes))

In [ ]:
removed = sorted(set(teams) - set(df["home_team"]) - set(df["away_team"]))
print(f"{len(removed)} teams removed, e.g.:", ", ".join(removed[:15]), "...")
df["period"].value_counts().reindex(["warm-up", "train", "test", "after"]).to_frame("matches")

> **From here on, the exploration only uses the training period** (plus the warm-up history where noted). The test period — including the 90 CONMEBOL qualifiers of the 2026 cycle — is not looked at until the final evaluation, so it can't influence our choices.

In [ ]:
train = df[df["period"] == "train"].copy()
train["total_goals"] = train["home_score"] + train["away_score"]
train["margin"] = train["home_score"] - train["away_score"]
print(f"Training matches: {len(train):,} | teams: {pd.unique(train[['home_team', 'away_team']].values.ravel()).size}")

### 3.3 The targets: home and away goals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
bins = list(range(0, 10))
labels = [str(b) for b in bins[:-1]] + ["9+"]
counts = pd.DataFrame({
    side: train[col].clip(upper=9).value_counts(normalize=True).reindex(bins, fill_value=0).values
    for side, col in [("Home goals", "home_score"), ("Away goals", "away_score")]
}, index=labels)
counts.plot(kind="bar", ax=ax, rot=0, width=0.8)
ax.set(title="Goals per team in a match (training period)", xlabel="Goals", ylabel="Share of matches")
plt.tight_layout(); plt.show()

train[TARGETS].agg(["mean", "var", "median", "max"]).T.round(2)

In [ ]:
print("Matches won by 10 or more goals:", (train["margin"].abs() >= 10).sum())
train.loc[train["margin"].abs().sort_values(ascending=False).index[:5],
          ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]]

### 3.4 Home advantage

In [ ]:
def outcome_table(d):
    return pd.Series({
        "matches": len(d),
        "home goals": d["home_score"].mean(),
        "away goals": d["away_score"].mean(),
        "home win %": 100 * (d["margin"] > 0).mean(),
        "draw %": 100 * (d["margin"] == 0).mean(),
        "away win %": 100 * (d["margin"] < 0).mean(),
    })

venue = train.groupby(train["neutral"].map({False: "home venue", True: "neutral venue"})).apply(outcome_table)
ax = venue[["home win %", "draw %", "away win %"]].plot(kind="bar", rot=0, figsize=(7, 3.5))
ax.set(title="Results by venue (training period)", xlabel="", ylabel="% of matches")
plt.tight_layout(); plt.show()
venue.round(2)

### 3.5 Match importance

In [ ]:
importance = train.groupby("tournament_group", observed=True).apply(
    lambda d: pd.Series({"matches": len(d), "goals per match": d["total_goals"].mean(),
                         "mean |margin|": d["margin"].abs().mean(),
                         "home win %": 100 * (d["margin"] > 0).mean()}))
importance.round(2)

### 3.6 CONMEBOL World Cup qualifiers (training period)

Since the qualifiers for the 1998 World Cup, CONMEBOL plays one league: every team meets every other team home and away.

In [ ]:
wcq = train[train["is_conmebol_wcq"] & (train["date"] >= "1996-01-01")].copy()
cycle_end = {1997: 1998, 2001: 2002, 2005: 2006, 2009: 2010, 2013: 2014, 2017: 2018, 2022: 2022}
wcq["cycle"] = wcq["date"].dt.year.map(lambda y: next(wc for end, wc in cycle_end.items() if y <= end))
print(wcq.groupby("cycle").size().rename("matches").to_string())
print(f"\nAll CONMEBOL qualifiers 1996-2022: {len(wcq)}")
outcome_table(wcq).round(2)

In [ ]:
rows = []
for team in CONMEBOL:
    home = wcq[wcq["home_team"] == team]
    away = wcq[wcq["away_team"] == team]
    home_pts = 3 * (home["margin"] > 0) + (home["margin"] == 0)
    away_pts = 3 * (away["margin"] < 0) + (away["margin"] == 0)
    rows.append({"team": team, "played": len(home) + len(away),
                 "points per game, home": home_pts.mean(), "points per game, away": away_pts.mean(),
                 "goals for per game": (home["home_score"].sum() + away["away_score"].sum()) / (len(home) + len(away)),
                 "goals against per game": (home["away_score"].sum() + away["home_score"].sum()) / (len(home) + len(away))})
teams_wcq = pd.DataFrame(rows).set_index("team").sort_values("points per game, home", ascending=False)

ax = teams_wcq[["points per game, home", "points per game, away"]].plot(kind="bar", rot=0, figsize=(11, 3.8))
ax.set(title="CONMEBOL qualifiers 1996-2022: points per game at home and away", xlabel="", ylabel="Points per game")
plt.tight_layout(); plt.show()
teams_wcq.round(2)

### 3.7 Altitude

Bolivia plays its home qualifiers in La Paz (≈ 3,640 m) and recently El Alto (≈ 4,100 m); Ecuador mostly in Quito (≈ 2,850 m); Colombia sometimes in Bogotá (≈ 2,640 m). Do visiting teams struggle there?

In [ ]:
HIGH_ALTITUDE_M = {"La Paz": 3640, "El Alto": 4100, "Quito": 2850, "Bogotá": 2640}
wcq["venue"] = np.where(wcq["city"].isin(["La Paz", "El Alto"]), "Bolivia: La Paz / El Alto",
               np.where(wcq["city"].isin(["Quito", "Bogotá"]), "Quito / Bogotá", "other venues"))
altitude = wcq.groupby("venue").apply(outcome_table)
altitude.round(2)

In [ ]:
# Same team, different altitude: Bolivia and Ecuador at home vs away
for team in ["Bolivia", "Ecuador"]:
    print(f"{team}: points per game at home {teams_wcq.loc[team, 'points per game, home']:.2f}, "
          f"away {teams_wcq.loc[team, 'points per game, away']:.2f}")

### 3.8 Main observations

- **Data quality.** 49,945 matches from 1872 to Oct 2026 with no missing values, no duplicated rows and no impossible scores. Removing regional and unrecognised teams dropped 2,565 matches; the training period then has **25,266 matches between 211 national teams**. The dip in 2020 is the COVID-19 pandemic.
- **The targets are small, skewed counts.** Home teams score 1.62 goals on average and away teams 1.08, with a median of 1 for both. 26% of home teams and 40% of away teams score 0. The variance (2.73 and 1.65) is larger than the mean — a long tail of big wins. 101 training matches were won by 10 or more goals (e.g. Australia 31–0 American Samoa); they are real, official matches, so we keep them, but they will push up the RMSE of every model. Team-strength features should explain most of them.
- **Home advantage is large.** At a home venue the home team scores 1.67 goals against 1.00 and wins 50.8% of the matches (away wins 25.7%). On neutral ground the gap almost disappears (1.49 vs 1.29). → `neutral` must be a feature.
- **Match importance matters.** Qualifiers have the most lopsided results (mean margin 1.84 goals, against 1.37 in friendlies), because they include mismatches between strong and very weak teams. → a match-importance feature.
- **CONMEBOL qualifiers 1996–2022:** 593 matches in 7 cycles (72 when Brazil qualified automatically for 1998 and as hosts for 2014; 89 for 2022 because Brazil vs Argentina in September 2021 was suspended). The home team wins 54.0% (draw 24.3%, away win 21.7%) and scores 1.67 goals against 0.95 — a **stronger home advantage than in international football overall**.
- **Brazil and Argentina** are the strongest teams both at home and away (2.59 / 1.56 and 2.30 / 1.57 points per game).
- **Altitude.** Bolivia takes 1.59 points per game at home but only 0.13 away; Ecuador 2.13 against 0.72 — by far the biggest home/away gaps. In Quito and Bogotá the home team wins 59.7% and visitors score only 0.72 goals per game. In La Paz / El Alto the home win rate is lower (42.6%) because Bolivia is a weaker team overall, but compared with Bolivia's away record the effect is huge. → we move the **altitude** feature from "stretch" to the planned features of Phase 2.

**What this means for Phase 2:** team strength (Elo ratings) to explain the big wins and the dominance of Brazil and Argentina, recent form, the `neutral` flag, a match-importance feature, and venue altitude.

## 4. Preprocess the data — feature engineering

The raw table only says *who* played *where* and *what the score was*. A model needs to know, **before kick-off**, how strong both teams are, how they have played recently, how they did against each other, and where the match is played. All features are built in `src/features.py`:

| Feature | For each match | Why (from section 3) |
|---|---|---|
| `elo_home`, `elo_away` | Elo rating of both teams | Team strength — explains the big wins and the dominance of Brazil and Argentina |
| `form_gf_*`, `form_ga_*`, `form_pts_*` | Goals for, goals against and points per match over each team's last 10 matches | Current form, which ratings react to only slowly |
| `h2h_gd`, `h2h_n` | Mean goal difference (home-team view) in the last 5 meetings, and how many meetings there were | Some match-ups have a history of their own |
| `neutral` | 1 if the venue is neutral | Home advantage is ≈ 0.7 goals at a home venue and almost gone on neutral ground |
| `altitude_km` | Venue altitude for venues at 1,500 m or higher (La Paz 3.64 km, Quito 2.85 km, …), else 0 | Bolivia and Ecuador have the largest home/away gaps |
| `is_world_cup`, `is_continental_final`, `is_qualifier`, `is_other_tournament` | Tournament group (Friendly = reference) | Qualifiers have the most lopsided scores |
| `year` | Calendar year | Long-term trends in scoring |

> **The leakage rule.** Matches are processed in date order. For every match, the features are read **first**, and only **then** is the match's result added to the team statistics. So no match can see its own result or any later result. `tests/test_features.py` checks this: changing the score of one match must not change the features of that match or of any earlier match.

**Elo ratings** (World Football Elo method, eloratings.net). Every team starts at 1500. After each match:

$$R_\text{new} = R_\text{old} + K \cdot G \cdot (W - W_e), \qquad W_e = \frac{1}{10^{-dr/400} + 1}$$

- $W$ = 1 for a win, 0.5 for a draw, 0 for a loss; $W_e$ = the expected result, where $dr$ is the rating difference **+100 for the home team** (unless the venue is neutral).
- $K$ = match importance: 60 World Cup · 50 continental final · 40 qualifier / Nations League · 30 other tournament · 20 friendly.
- $G$ = margin of victory: 1 for 0–1 goals, 1.5 for 2 goals, $(11 + N)/8$ for $N \geq 3$.

The ratings are computed over the **whole history since 1872**, so they are already meaningful when the training period starts in 1994.

In [ ]:
feat = build_features(df)
data = df.drop(columns="neutral").join(feat)          # the features' 0/1 version of `neutral` is kept
print(f"Feature table: {len(feat):,} matches x {len(FEATURES)} model features (built in date order)")
data.loc[data["period"] == "train", ["date", "home_team", "away_team", "home_score", "away_score"] + FEATURES].tail(3)

### 4.1 Do the Elo ratings make sense?

In [ ]:
train_data = data[data["period"] == "train"]

# pre-match rating of every CONMEBOL team over the training period
long = pd.concat([
    train_data[["date", "home_team", "elo_home"]].set_axis(["date", "team", "elo"], axis=1),
    train_data[["date", "away_team", "elo_away"]].set_axis(["date", "team", "elo"], axis=1),
])
long = long[long["team"].isin(CONMEBOL)].sort_values("date")

fig, ax = plt.subplots(figsize=(12, 4.5))
for team, d in long.groupby("team"):
    ax.plot(d["date"], d["elo"].rolling(5, min_periods=1).mean(), label=team, lw=1.4)
ax.set(title="Elo rating of the CONMEBOL teams, 1994 – Aug 2023 (pre-match, smoothed over 5 matches)",
       xlabel="", ylabel="Elo rating")
ax.legend(ncol=5, fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

In [ ]:
# each team's pre-match rating in its last training match (end of August 2023)
last_home = train_data[["date", "home_team", "elo_home"]].set_axis(["date", "team", "elo"], axis=1)
last_away = train_data[["date", "away_team", "elo_away"]].set_axis(["date", "team", "elo"], axis=1)
latest = (pd.concat([last_home, last_away]).sort_values("date", kind="stable")
            .groupby("team").tail(1).set_index("team")["elo"].sort_values(ascending=False))
print("Top 10 at the end of the training period:")
print(latest.head(10).round(0).to_string())
print("\nCONMEBOL teams:")
print(latest[CONMEBOL].sort_values(ascending=False).round(0).to_string())

In [ ]:
bands = pd.cut(train_data["elo_diff"], [-2000, -200, -100, 0, 100, 200, 2000],
               labels=["< -200", "-200 to -100", "-100 to 0", "0 to 100", "100 to 200", "> 200"])
home_win = (train_data["home_score"] > train_data["away_score"]).groupby(bands, observed=True).mean() * 100
ax = home_win.plot(kind="bar", rot=0, figsize=(8, 3.5), color="tab:blue")
ax.set(title="Home win % by pre-match Elo difference (home − away), training period",
       xlabel="Elo difference", ylabel="Home win %")
for i, v in enumerate(home_win):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.tight_layout(); plt.show()

### 4.2 How strongly is each feature related to the targets?

In [ ]:
corr = pd.DataFrame({target: train_data[FEATURES + ["elo_diff"]].corrwith(train_data[target])
                     for target in TARGETS})
ax = corr.sort_values("home_score").plot(kind="barh", figsize=(8, 6))
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Correlation with home and away goals (training period)", xlabel="Pearson correlation")
plt.tight_layout(); plt.show()
corr.round(3).sort_values("home_score", ascending=False)

### 4.3 Altitude

In [ ]:
at_home = train_data[train_data["neutral"] == 0]
high = at_home["altitude_km"] > 0
print(f"Training matches at a venue of 1,500 m or higher: {high.sum():,} "
      f"(test period: {(data.loc[data['period'] == 'test', 'altitude_km'] > 0).sum()})")
pd.DataFrame({
    "matches": [high.sum(), (~high).sum()],
    "home goals": [at_home.loc[high, "home_score"].mean(), at_home.loc[~high, "home_score"].mean()],
    "away goals": [at_home.loc[high, "away_score"].mean(), at_home.loc[~high, "away_score"].mean()],
    "home win %": [100 * (at_home.loc[high, "home_score"] > at_home.loc[high, "away_score"]).mean(),
                   100 * (at_home.loc[~high, "home_score"] > at_home.loc[~high, "away_score"]).mean()],
}, index=["venue ≥ 1,500 m", "lower venues"]).round(2)

### 4.4 Missing values in the features

In [ ]:
missing = data.loc[data["period"].isin(["train", "test"]), FEATURES].isna()
print("Missing values per period:")
print(missing.groupby(data["period"], observed=True).sum().loc[:, lambda d: d.sum() > 0])
data.loc[missing.index[missing.any(axis=1)], ["date", "home_team", "away_team", "period", "n_prev_home", "n_prev_away"]]

### 4.5 Feature observations

- **The Elo ratings behave as they should.** The home team wins 14% of the time when it is rated more than 200 points lower, and 81% when it is rated more than 200 points higher, with a steady rise in between. At the end of the training period (Aug 2023) the top three are Argentina (2202), Brazil (2180) and France (2163) — the 2022 world champion and the two other strongest teams of that time. Among the CONMEBOL teams Bolivia is lowest (1677), and the chart shows known stories such as Venezuela's rise from about 1,300 around 2000 to about 1,800.
- **The strongest single signal is the rating difference** (r = 0.46 with home goals, −0.42 with away goals), followed by the head-to-head record (0.36 / −0.32) and how many goals the *opponent* has been conceding (0.35 / 0.32). `elo_diff` itself is not a model feature, because it is exactly `elo_home − elo_away` and would make the normal equation unsolvable; a linear model can rebuild it from the two ratings.
- **`neutral`, the tournament dummies, `year` and `altitude_km` are hardly correlated with the goals on their own.** That doesn't make them useless: home advantage, for example, works *together* with team strength. Whether they help will show in the models (sections 7–10).
- **Altitude:** 675 training matches were played at venues of 1,500 m or higher. There, away teams score 0.85 goals per match against 1.01 elsewhere, and the home team wins 53.3% against 50.7%. Only 4% of matches are at altitude, which is why the overall correlation is close to 0.
- **Missing values:** 9 training matches have no recent form, because they are the **first matches of new national teams** (e.g. Czech Republic 1994, Kosovo 2014, South Sudan 2012). The test period has none. They are filled with training medians in section 6. A related limitation: successor states such as the Czech Republic start at Elo 1500 instead of inheriting their predecessor's rating.

## 5. Define features and target

- **Targets (y):** `home_score` and `away_score` — two numbers per match. Both algorithms predict them together (our scratch model fits one coefficient column per target; the random forest supports several targets natively).
- **Features (X):** the 17 pre-match features from section 4. They are all numeric already, so no encoding is needed; scaling and filling the few missing form values happen in section 6, fitted on the training data only.

In [ ]:
FEATURE_DESCRIPTIONS = {
    "elo_home": "Elo rating of the home team before the match",
    "elo_away": "Elo rating of the away team before the match",
    "form_gf_home": f"Home team: goals scored per match, last {FORM_WINDOW} matches",
    "form_ga_home": f"Home team: goals conceded per match, last {FORM_WINDOW} matches",
    "form_pts_home": f"Home team: points per match (3/1/0), last {FORM_WINDOW} matches",
    "form_gf_away": f"Away team: goals scored per match, last {FORM_WINDOW} matches",
    "form_ga_away": f"Away team: goals conceded per match, last {FORM_WINDOW} matches",
    "form_pts_away": f"Away team: points per match (3/1/0), last {FORM_WINDOW} matches",
    "h2h_gd": f"Mean goal difference (home view) in the last {H2H_WINDOW} meetings, 0 if none",
    "h2h_n": f"Number of earlier meetings counted (0-{H2H_WINDOW})",
    "neutral": "1 if played on neutral ground",
    "altitude_km": "Venue altitude in km for venues at 1,500 m or higher, else 0",
    "is_world_cup": "1 if a World Cup match",
    "is_continental_final": "1 if a continental championship match (Copa América, Euro, ...)",
    "is_qualifier": "1 if a qualifier or Nations League match",
    "is_other_tournament": "1 if another tournament (friendly = all four are 0)",
    "year": "Calendar year of the match",
}
assert list(FEATURE_DESCRIPTIONS) == FEATURES

X = data[FEATURES]
y = data[TARGETS]
print("X:", X.shape, "| y:", y.shape)
pd.Series(FEATURE_DESCRIPTIONS, name="meaning").to_frame()

In [ ]:
# save the feature table so later phases can start from it without rebuilding (git-ignored)
out = ROOT / "data" / "processed" / "features.csv"
data.to_csv(out, index=False)
print(f"Saved {out.relative_to(ROOT)} ({len(data):,} rows)")

## 6. Split into training and test sets

The matches are a **time series**: a team's strength in 2024 is not something we could have known in 2010. A random 80/20 split would put matches from 2024 in the training set and matches from 2010 in the test set, so the model would partly learn from the future and its test score would be too optimistic. We therefore split **by date**:

| Set | Period | Used for |
|---|---|---|
| **Training** | 1 Jan 1994 → 31 Aug 2023 | fitting both models; tuning with time-series cross-validation |
| **Test — all** | 1 Sep 2023 → 9 Sep 2025 | final evaluation on all matches between FIFA teams in the 2026 qualifying period |
| **Test — CONMEBOL** | the 90 CONMEBOL World Cup qualifiers inside that window | the main evaluation for our topic |

**Missing values.** The 9 training matches without recent form (debut matches of new teams, section 4.4) are filled with the **training medians**. The same medians are used for the test set, so nothing is learned from it.

In [ ]:
train_df = data[data["period"] == "train"].reset_index(drop=True)
test_df = data[data["period"] == "test"].reset_index(drop=True)

X_train, y_train = train_df[FEATURES], train_df[TARGETS]
X_test, y_test = test_df[FEATURES], test_df[TARGETS]
conmebol_test = test_df["is_conmebol_wcq"].to_numpy()

train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)            # training medians - nothing is learned from the test set

pd.DataFrame({
    "matches": [len(train_df), len(test_df), conmebol_test.sum()],
    "from": [train_df["date"].min().date(), test_df["date"].min().date(), test_df.loc[conmebol_test, "date"].min().date()],
    "to": [train_df["date"].max().date(), test_df["date"].max().date(), test_df.loc[conmebol_test, "date"].max().date()],
    "missing values": [int(X_train.isna().sum().sum()), int(X_test.isna().sum().sum()), int(X_test[conmebol_test].isna().sum().sum())],
}, index=["training", "test - all", "test - CONMEBOL qualifiers"])

**Validation inside the training set.** For comparing settings and tuning (sections 7 and 8) we use `TimeSeriesSplit` with 5 folds: every fold trains on all matches up to a date and validates on the period right after it, so the model never trains on the future.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
folds = [{"fold": k + 1,
          "train until": train_df.loc[tr[-1], "date"].date(),
          "validate from": train_df.loc[va[0], "date"].date(),
          "validate until": train_df.loc[va[-1], "date"].date(),
          "train matches": len(tr), "validation matches": len(va)}
         for k, (tr, va) in enumerate(tscv.split(X_train))]
pd.DataFrame(folds).set_index("fold")

## 7. Algorithm 1 — Linear Regression from scratch

As the guidelines require, this model is **implemented by us with NumPy only** — no regression model from scikit-learn or another library. The class lives in `src/linear_regression.py` so that it can be tested (`tests/test_linear_regression.py`), and its complete code is shown below.

### 7.1 How it works

**Model.** For a match with feature vector $x$ (17 features), the predicted home and away goals are a weighted sum of the features plus an intercept:

$$\hat{y}_\text{home} = \theta_{0,h} + \sum_j \theta_{j,h}\, x_j \qquad \hat{y}_\text{away} = \theta_{0,a} + \sum_j \theta_{j,a}\, x_j$$

In matrix form $\hat{Y} = X_b\,\theta$, where $X_b$ is the feature matrix with a column of 1s added for the intercept, and $\theta$ has **one column per target**. Fitting both columns at once gives exactly the same result as two separate regressions on the same features (checked in the tests).

**Cost.** The mean squared error, plus an optional L2 (ridge) penalty on the coefficients — never on the intercept:

$$J(\theta) = \frac{1}{m}\sum_{i=1}^{m}\big(\hat{y}_i - y_i\big)^2 + \lambda \sum_{j\geq 1} \theta_j^2$$

**Fitting, way 1 — the normal equation** (closed form). Setting the gradient of $J$ to zero gives

$$\big(X_b^\top X_b + m\lambda I^*\big)\,\theta = X_b^\top Y$$

where $I^*$ is the identity matrix with a 0 for the intercept. We solve this linear system with `np.linalg.solve` instead of computing a matrix inverse, which is faster and numerically more stable.

**Fitting, way 2 — batch gradient descent.** Start with $\theta = 0$ and repeatedly step downhill:

$$\theta \leftarrow \theta - \eta \Big[\tfrac{2}{m} X_b^\top (X_b\theta - Y) + 2\lambda\,\theta^*\Big]$$

with learning rate $\eta$ and $\theta^*$ = θ with the intercept row set to 0. It stops after `n_iters` steps or when the cost improves by less than `tol`. The cost is convex, so gradient descent cannot get stuck in a local minimum and must end at the same θ as the normal equation.

**Standardisation.** The features are on very different scales (Elo ≈ 1,500, dummies 0/1, altitude 0–4). Before fitting, every feature is standardised with the **training** mean and standard deviation (stored in `fit`, reused in `predict`). Gradient descent needs this — otherwise the cost surface is a long narrow valley and the steps barely move — and it makes the coefficients comparable with each other.

### 7.2 The code

In [ ]:
Code(inspect.getsource(LinearRegressionScratch), language="python")

**What the tests check** (`tests/test_linear_regression.py`, run with `pytest`): the normal equation recovers a known θ exactly; the predictions equal NumPy's least-squares solution (`np.linalg.lstsq`, plain linear algebra); gradient descent converges to the normal-equation θ, with and without L2; the cost never increases during gradient descent; a too-large learning rate is detected; two targets at once equal two separate fits; L2 shrinks the coefficients but not the intercept; missing values are rejected.

### 7.3 Fitting on the training data — both ways

In [ ]:
start = time.perf_counter()
lr_normal = LinearRegressionScratch(method="normal").fit(X_train, y_train)
t_normal = time.perf_counter() - start

start = time.perf_counter()
lr_gd = LinearRegressionScratch(method="gradient", learning_rate=0.1, n_iters=20_000, tol=1e-12).fit(X_train, y_train)
t_gd = time.perf_counter() - start

print(f"Normal equation : {t_normal:.3f} s")
print(f"Gradient descent: {t_gd:.1f} s, {lr_gd.n_iter_} iterations, final cost {lr_gd.loss_history_[-1]:.5f}")
print(f"Largest difference between the two θ: {np.abs(lr_gd.theta_ - lr_normal.theta_).max():.1e}")

### 7.4 Choosing the learning rate

How large can the learning rate be? For this cost, gradient descent is stable only if $\eta < 1/\lambda_\max$, where $\lambda_\max$ is the largest eigenvalue of $X_b^\top X_b / m$ (computed on the standardised training features).

In [ ]:
Xs = (X_train - X_train.mean()) / X_train.std(ddof=0)
Xb = np.column_stack([np.ones(len(Xs)), Xs])
eigenvalues = np.linalg.eigvalsh(Xb.T @ Xb / len(Xb))
print(f"Largest eigenvalue {eigenvalues.max():.3f} -> stable for a learning rate below {1 / eigenvalues.max():.3f}")
print(f"Condition number {eigenvalues.max() / eigenvalues.min():.0f} (small = gradient descent converges quickly)")

optimum = lr_normal._cost(Xb, y_train.to_numpy(float), lr_normal.theta_)
fig, ax = plt.subplots(figsize=(9, 4))
for eta in [0.01, 0.05, 0.1, 0.2]:
    gd = LinearRegressionScratch(method="gradient", learning_rate=eta, n_iters=1500, tol=0).fit(X_train, y_train)
    # below ~1e-12 the difference is only floating-point rounding, so the curve is cut there
    ax.plot(np.clip(np.array(gd.loss_history_) - optimum, 1e-12, None), label=f"η = {eta}")
ax.set(yscale="log", ylim=(5e-13, 10), xlabel="Iteration", ylabel="Cost − optimal cost (log scale)",
       title="Gradient descent on the training data: distance to the normal-equation optimum")
ax.legend()
plt.tight_layout(); plt.show()

try:
    LinearRegressionScratch(method="gradient", learning_rate=0.3, n_iters=2000).fit(X_train, y_train)
except RuntimeError as error:
    print("η = 0.3:", error)

### 7.5 The learned coefficients

In [ ]:
coefficients = pd.DataFrame(lr_normal.coef_, index=FEATURES, columns=["home goals", "away goals"])
print("Intercepts (= mean goals, because the features are standardised):", lr_normal.intercept_.round(3))

ax = coefficients.sort_values("home goals").plot(kind="barh", figsize=(8, 6))
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Coefficients per standard deviation of each feature", xlabel="Change in predicted goals")
plt.tight_layout(); plt.show()
coefficients.round(3).sort_values("home goals", ascending=False)

In [ ]:
def fit_scores(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return pd.DataFrame({
        "MAE": np.abs(y_true - y_pred).mean(axis=0),
        "RMSE": np.sqrt(((y_true - y_pred) ** 2).mean(axis=0)),
        "R²": 1 - ((y_true - y_pred) ** 2).sum(axis=0) / ((y_true - y_true.mean(axis=0)) ** 2).sum(axis=0),
    }, index=TARGETS)

print("Fit on the TRAINING data (the test set is evaluated in section 9):")
fit_scores(y_train, lr_normal.predict(X_train)).round(3)

### 7.6 Observations

- **Both ways of fitting give the same model.** The largest difference between the gradient-descent θ and the normal-equation θ is about 7 · 10⁻⁶. The normal equation takes a few hundredths of a second; gradient descent needs 414 iterations (about a second) with η = 0.1. With only 17 features the normal equation is the practical choice; gradient descent is what would scale to many more features.
- **The learning rate behaves as the theory says.** The largest eigenvalue of $X_b^\top X_b/m$ is 3.70, so gradient descent is stable only for η < 0.27 — and η = 0.3 indeed diverges after 49 iterations. η = 0.2 converges fastest, η = 0.01 is still far from the optimum after 1,500 iterations. We use η = 0.1, which leaves a safety margin. The condition number is only 28 because the features are standardised; without standardisation gradient descent would hardly move.
- **What the model learned** (converted back to the original units):
  - **Team strength dominates.** A home team rated 100 Elo points higher scores about 0.25 more goals and concedes about 0.16 fewer; the same holds the other way round for the away team.
  - **The opponent's defence matters.** Every extra goal per match the away team has been conceding adds about 0.51 goals to the predicted home score (and the home team's conceding adds 0.44 to the away score).
  - **Home advantage:** on neutral ground the home team scores about 0.20 fewer goals and the away team 0.29 more — together about half a goal of goal difference.
  - **Altitude:** each kilometre adds about 0.06 home goals and removes about 0.10 away goals — in La Paz (3.64 km) about +0.21 and −0.37 compared with sea level. It is small overall, but it points in the direction section 3.7 suggested.
- **Not every coefficient can be read on its own.** `form_pts_home` gets a *negative* coefficient for home goals, although more points should mean more goals. It is strongly correlated with the home team's Elo (r = 0.68), goals scored (0.73) and goals conceded (−0.69), so the model splits their shared effect between them (**multicollinearity**). The predictions are fine; the individual coefficients of correlated features are not.
- **Fit on the training data:** R² = 0.28 for home goals and 0.23 for away goals, MAE 1.04 and 0.84 goals. That is modest but expected — goals are rare, noisy events. Whether the model beats simple baselines on unseen matches is the real test (section 9).

### 7.7 Tuning λ with time-series cross-validation

The only hyperparameter of our model that changes the result is the L2 strength λ (the learning rate only changes how fast gradient descent gets there). We compare λ = 0, 0.001, 0.01, 0.1, 1 and 10 with the 5 time-ordered folds from section 6: for every fold a **fresh model** is fitted on the earlier matches and scored on the following period (RMSE averaged over home and away goals). Our own loop (`time_series_cv` in `src/evaluation.py`) is used, because our model is not a scikit-learn estimator.

In [ ]:
L2_GRID = [0, 0.001, 0.01, 0.1, 1, 10]
cv_rmse = pd.DataFrame({l2: time_series_cv(lambda: LinearRegressionScratch(l2=l2), X_train, y_train)
                        for l2 in L2_GRID}, index=[f"fold {k}" for k in range(1, 6)]).T
cv_rmse.index.name = "λ"
cv_rmse["mean"] = cv_rmse.mean(axis=1)
best_l2 = float(cv_rmse["mean"].idxmin())
print(f"Best λ: {best_l2}  (mean validation RMSE {cv_rmse.loc[best_l2, 'mean']:.4f}, "
      f"λ = 0: {cv_rmse.loc[0, 'mean']:.4f})")
cv_rmse.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot([str(l2) for l2 in L2_GRID], cv_rmse["mean"], marker="o")
ax.set(title="Mean validation RMSE per λ (5 time-series folds)", xlabel="λ", ylabel="RMSE (goals)")
plt.tight_layout(); plt.show()

lr_final = LinearRegressionScratch(method="normal", l2=best_l2).fit(X_train, y_train)
print("Final model:", lr_final)

**Result.** λ = 0.01 gives the lowest validation error, but the gain over no regularisation is tiny (RMSE 1.2528 against 1.2530); from λ = 0.1 upwards the model gets worse. Regularisation fights overfitting, and a linear model with 17 features and 25,000 training matches hardly overfits (the same thing happened with Ridge in Lab 1). We keep λ = 0.01 because cross-validation chose it. The first fold (validation 1999–2004) has the highest error in every row — more lopsided results in that period.

## 8. Algorithm 2 — Random Forest Regressor

_To do — Phase 5: grid search with `TimeSeriesSplit`._

## 9. Evaluate the models

All models are fitted on the **training period only** and evaluated on the two test sets from section 6. We compare them with two baselines that use no features at all:

| Baseline | Prediction |
|---|---|
| **Mean** | The average home and away goals of the training period (1.62 – 1.08) for every match |
| **Venue** | The training averages separately for home venues (1.67 – 1.00) and neutral venues (1.49 – 1.29) |

A model is only useful if it beats these. Metrics (`src/evaluation.py`): **MAE**, **MSE**, **RMSE** and **R²** per target and averaged over home and away goals. Note that R² on a test set compares the model with the mean *of that test set* — a number no model can know in advance — so on the small CONMEBOL set it can be negative even for a reasonable model; the comparison with the baselines is the fairer test.

In [ ]:
models = {
    "Mean baseline": MeanBaseline().fit(X_train, y_train),
    "Venue baseline": VenueBaseline().fit(X_train, y_train),
    "Linear Regression (scratch)": lr_final,
}
# Phase 5 adds the random forest to this dictionary.

test_sets = {"all test matches": np.ones(len(test_df), dtype=bool), "CONMEBOL qualifiers": conmebol_test}

rows = []
for name, model in models.items():
    for set_name, mask in test_sets.items():
        table = regression_metrics(y_test[mask], model.predict(X_test[mask]))
        for target, values in table.iterrows():
            rows.append({"test set": set_name, "target": target, "model": name, **values})
results = pd.DataFrame(rows).set_index(["test set", "target", "model"])

print("Average over home and away goals:")
results.xs("both (average)", level="target").round(3)

In [ ]:
print("Per target:")
results.drop(index="both (average)", level="target").round(3)

### 9.1 Win / draw / loss and exact scores

The models predict goals, but the result of a match is what people care about. A predicted home win is `predicted home goals − predicted away goals > band`, a draw when the difference is smaller than the band. A regression model almost never predicts exactly equal goals, so the **draw band** is chosen per model to maximise the win/draw/loss accuracy **on the training data**. For the exact score the predictions are rounded (and negative values set to 0).

In [ ]:
draw_bands = {name: choose_draw_band(y_train, model.predict(X_train)) for name, model in models.items()}
outcome_rows = []
for name, model in models.items():
    for set_name, mask in test_sets.items():
        scores = outcome_scores(y_test[mask], model.predict(X_test[mask]), draw_bands[name])
        outcome_rows.append({"test set": set_name, "model": name, "draw band": draw_bands[name], **scores})
outcome_table = pd.DataFrame(outcome_rows).set_index(["test set", "model"])

actual = test_df.loc[conmebol_test, "home_score"] - test_df.loc[conmebol_test, "away_score"]
print(f"Actual results of the 90 CONMEBOL qualifiers: home win {(actual > 0).mean():.1%}, "
      f"draw {(actual == 0).mean():.1%}, away win {(actual < 0).mean():.1%}")
outcome_table.round(3)

### 9.2 The 2026 CONMEBOL qualifiers, match by match

In [ ]:
def prediction_table(model, mask):
    p = model.predict(X_test[mask])
    t = test_df.loc[mask, ["date", "home_team", "away_team", "city", "home_score", "away_score"]].copy()
    t["predicted"] = [f"{h:.1f} – {a:.1f}" for h, a in p]
    t["actual"] = t["home_score"].astype(str) + " – " + t["away_score"].astype(str)
    t["abs error"] = np.abs(p - t[["home_score", "away_score"]].to_numpy()).sum(axis=1)
    return t.drop(columns=["home_score", "away_score"])

conmebol_pred = prediction_table(lr_final, conmebol_test)
print("Linear Regression (scratch) - first round of the cycle:")
conmebol_pred.head(10)

In [ ]:
print("Largest errors (home + away goals) in the CONMEBOL test set:")
conmebol_pred.sort_values("abs error", ascending=False).head(8)

In [ ]:
print("Bolivia's home matches (La Paz 3,640 m, El Alto 4,090 m):")
conmebol_pred[conmebol_pred["home_team"] == "Bolivia"]

### 9.3 Observations — Algorithm 1 against the baselines

**All 2,269 test matches.** Our linear regression is clearly better than both baselines on every metric: average MAE 0.93 against 1.07 goals, RMSE 1.23 against 1.43, and R² 0.29 for home goals and 0.23 for away goals — almost exactly the training values (0.28 and 0.23), so the model **generalises to 2023–2025 without overfitting**. Its average predictions are well calibrated (home 1.58 predicted vs 1.63 actual, away 1.10 vs 1.13). It gets the result right in 59.9% of the matches against 47.0% for the baselines (which always predict a home win), and the exact score in 11.3% against 6.8–8.7%.

**The 90 CONMEBOL qualifiers.** The model still beats the baselines on the goal errors — MAE 0.91 against 0.97 and RMSE 1.11 against 1.20 — but the gain is smaller, and on the match result it is *worse*: 50.0% correct against 54.4% for "always a home win". Three reasons:

1. **Visiting teams score far less than the model expects.** It predicts 0.98 away goals per match on average; the real number in this cycle was 0.60 (home goals are fine: 1.36 predicted vs 1.43). The model learned the *average* home advantage of international football, and the home advantage in South American qualifiers is much stronger (altitude, long journeys, climate, hostile stadiums — section 3.6 already showed 54% home wins and only 0.95 away goals before 2023). This is also why R² for away goals is negative here (−0.14): the model's level is too high compared with the mean of these 90 matches.
2. **It never predicts a draw.** On the training data no draw band improves the accuracy (the best band is 0), because draws are the hardest result to tell apart from narrow wins. In this cycle 27 of the 90 matches (30%) were draws, and all of them count as misses. Of the other matches it gets 35 of 49 home wins and 10 of 14 away wins right; 14 home wins were predicted as away wins.
3. **Bolivia in El Alto.** The largest errors are Bolivia's home matches at El Alto (4,090 m), where Bolivia beat Venezuela 4–0, Colombia 1–0, Chile 2–0 and Brazil 1–0 while the model predicted defeats. El Alto never appears in the training data (Bolivia moved there in 2024), Bolivia's Elo rating is the lowest in CONMEBOL, and the altitude effect the model learned (mostly from La Paz, Quito, Mexico City and Nairobi) is small and linear — it cannot capture an effect this strong at an altitude it has never seen.

**Other limitations.** 117 of the 2,269 test predictions are negative (strong teams playing away against very weak ones): a linear model has no lower bound at 0. They are only clipped for the exact-score check.

> **We do not fix these problems by changing the features now.** That would mean tuning the model on the test set, and the result would no longer be an honest estimate. They are discussed as limitations and future work instead (a competition-specific home advantage, a non-linear altitude effect, a model for count data such as Poisson regression, and a way to predict draws).

## 10. Compare the results

_To do — Phase 6._

## 11. Visualisations

_To do — Phase 6._